# Graphic design 01

Macro idea → designer’s proposal and image prompt → generated poster → same designer
examines the poster → revises its proposal or prompt → next poster.

The designer judges whether visible choices communicate the macro idea, using semiotic
reasoning where helpful. Its own proposal is provisional. No independent critic is required.


## 1. Setup

Launch Jupyter from the repository root or this experiment directory. Install the project with `python -m pip install -e ".[notebook]"` first.
Load local configuration without displaying secrets. Existing process variables take priority,
then `.env.local`, then `.env`. The generation cell runs a paid API call only when `RUN_GENERATION` is set to `True`.


In [1]:
import os
import json
from pathlib import Path
from dotenv import dotenv_values
from copy import deepcopy
from IPython.display import Markdown, display
from graphic_design_helper.workflow import generate_round, design_round, load_rounds
from graphic_design_helper.images import compare_images
from graphic_design_helper.records import save_experiment, start_run

# Locate the repository from either its root or an experiment directory.
REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src/graphic_design_helper").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Launch Jupyter inside the graphic-design-helper repository.")
EXPERIMENT_DIR = REPO_ROOT / "experiments" / "pilot1"
OUTPUT_DIR = EXPERIMENT_DIR / "outputs"

local_config = {
    **dotenv_values(REPO_ROOT / ".env"),
    **dotenv_values(REPO_ROOT / ".env.local"),
}
for key, value in local_config.items():
    if value is not None:
        os.environ.setdefault(key, value)
del local_config
from graphic_design_helper.prompt_builder import build_designer_request, request_token
from graphic_design_helper.workflow import review_token, compose_prompt

from graphic_design_helper.proposal_presentation import render_design_rationale


## 2. Design task and requirements

Specify the communication purpose, audience, viewing context, deliverable, and constraints
below. These values populate the corresponding sections of the designer input template.
The internal `brief` variable stores these requirements; no separate brief document is needed.
Optional additional wording can be supplied through `original_user_request`.

Use the experiment-selection cell below to start a new run or restore a saved one.
Restoring a run also restores its task wording and latest proposal. New run folders
are created only when a designer call is enabled.


In [2]:
original_user_request = None  # Preserve a separate user request verbatim when supplied.
brief = {
    "core_message": None,  # Unspecified: the designer must not treat an inference as confirmed.
    "topic": "Slowing down in everyday life",
    "purpose": "Invite people to reconsider pressure to remain continuously productive and allow a brief pause.",
    "audience": "Students or office workers feeling pressure to stay productive.",
    "setting": "A poster encountered briefly in a shared indoor space.",
    "tone": "Inviting rather than blaming or patronizing.",
    "visual_style": (
        "International Typographic Style (Swiss Style). "
        "Use a clear modular grid, asymmetric but balanced composition, "
        "sans-serif typography, and a strong typographic hierarchy. "
        "Use generous negative space, a predominantly black-and-white palette, "
        "and at most one restrained accent color. "
        "Let typography, spacing, and alignment carry the visual expression. "
        "Keep the composition quiet and spacious to support the invitation to pause. "
        "Avoid ornamental clutter, decorative effects, and unnecessary shadows."
    ),
    "constraints": "Avoid unsupported factual, health, or society-wide claims.",
    "deliverable": "A portrait poster in English. Propose the visible wording.",
    "exact_copy": None,  # Supply text here only if it is a genuine requirement.
}
previous_headline = "A moment, without a project."  # Not included in the request.
if "rounds" not in globals():
    rounds = []
if "proposal_records" not in globals():
    proposal_records = []

if "run_dir" not in globals():
    run_dir = None


### Select a new experiment or resume a saved one

Set `RESUME_RUN` to a folder name such as `"run_01"` to restore its latest completed
proposal without calling a model. Set it to `None` to prepare a new experiment;
the new directory is created only when the designer call is enabled.
Run this cell once when selecting an experiment. Running it again discards in-memory
drafts and reloads the saved state. After restarting the kernel, run Setup and this
section again. A page refresh alone does not reload Python modules or variables.

For a saved proposal awaiting its first image, leave `RUN_DESIGNER = False` and
continue to Section 5 to prepare, preview, and review the image request again.
For self-review of a generated poster, return to Section 3 and explicitly enable
the designer call after reviewing its input. Model-call switches default to `False`.


In [5]:
RESUME_RUN = None  # Start a new experiment.

if RESUME_RUN is not None:
    if not isinstance(RESUME_RUN, str) or Path(RESUME_RUN).name != RESUME_RUN:
        raise ValueError("Choose a run folder name directly under outputs.")
    restored_run = (OUTPUT_DIR / RESUME_RUN).resolve(strict=True)
    if restored_run.parent != OUTPUT_DIR.resolve():
        raise ValueError("The selected run must be directly under outputs.")
    manifest = json.loads((restored_run / "run.json").read_text(encoding="utf-8"))
    restored_input = json.loads((restored_run / "brief.json").read_text(encoding="utf-8"))
    restored_rounds = load_rounds(restored_run)
    restored_proposal = None
    if manifest.get("latest_designer_attempt"):
        attempt_dir = Path(manifest["latest_designer_attempt"]).resolve(strict=True)
        if not attempt_dir.is_relative_to(restored_run):
            raise ValueError("Saved proposal path is outside this run; repair its paths first.")
        restored_proposal = json.loads((attempt_dir / "response.json").read_text(encoding="utf-8"))
        if restored_proposal.get("status") != "completed":
            raise ValueError("The latest designer attempt is incomplete; inspect its record before continuing.")
        # Later attempts may include clarification added after the run began.
        restored_input = restored_proposal["designer_input"]
    run_dir = restored_run
    rounds = restored_rounds
    brief = deepcopy(restored_input["brief"])
    original_user_request = restored_input.get("original_user_request")
else:
    run_dir = None
    rounds = []
    restored_proposal = None

# Approval and preview state must be rebuilt from the selected experiment.
for stale_name in ("proposal_record", "image_spec", "designer_request", "active_round",
                   "image_prompt_preview", "image_preview_context", "designer_preview_token"):
    globals().pop(stale_name, None)
reviewed_token = None
proposal_records = []
if restored_proposal is not None:
    proposal_record = restored_proposal
    proposal_records.append(deepcopy(proposal_record))
    display(Markdown(render_design_rationale(proposal_record["proposal"])))
print(f"Experiment: {run_dir if run_dir is not None else 'new (not created yet)'}")
print(f"Saved image attempts: {len(rounds)}; proposal loaded: {restored_proposal is not None}")


Experiment: new (not created yet)
Saved image attempts: 0; proposal loaded: False


## 3. Preview the designer request

Initial proposals use `templates/designer-input-template.md`: the design task, audience,
requirements, design decisions, design approach, and proposal instructions.
Later rounds use `templates/designer-review-input-template.md`, which also receives the
previous proposal, the exact image prompt used, and the actual PNG as an image attachment.
The JSON response schema is shown separately because it is sent as the API's response
format. Editing instructions, schema, model settings, or the image requires a new preview.


In [6]:
designer_model = os.getenv("OPENAI_DESIGNER_MODEL", "").strip() or "gpt-5.6-luna"
designer_effort = "medium"

def current_designer_request():
    revision_context = None
    designer_image = None
    if rounds:
        previous = rounds[-1]
        if brief != previous["research"]["brief"]:
            raise ValueError("The macro brief changed. Start a new run for a new communication objective.")
        if not previous.get("image_path"):
            raise ValueError("The last attempt has no image. Resolve that attempt before continuing.")
        revision_context = {
            "parent_folder": previous["folder"],
            "previous_proposal": previous["research"].get("proposal"),
            "previous_image_spec": previous.get("image_spec"),
            "previous_prompt": previous.get("image_prompt") or previous.get("generation", {}).get("prompt") or previous.get("production_prompt"),
        }
        designer_image = previous["image_path"]
    return build_designer_request(
        brief, revision_context, original_user_request=original_user_request,
        image_path=designer_image, model=designer_model, reasoning_effort=designer_effort,
    )

if rounds and rounds[-1].get("status") in {"requested", "failed", "interrupted"}:
    designer_request = None
    designer_preview_token = None
    print("No image is available for self-review. For a failed/interrupted call, continue to Section 5 and explicitly retry in Section 6.")
else:
    designer_request = current_designer_request()
    designer_preview_token = request_token(designer_request)
    print(designer_request["prompt"])
    print(json.dumps({key: value for key, value in designer_request.items()
                      if key not in {"prompt", "designer_input"}}, ensure_ascii=False, indent=2))
    if designer_request.get("image_path"):
        display(compare_images([designer_request["image_path"]], ["Poster for self-review"]))


# Design task

## Topic

Slowing down in everyday life

## Communication purpose

Invite people to reconsider pressure to remain continuously productive and allow a brief pause.

## Core message

Not specified.

## Deliverable

A portrait poster in English. Propose the visible wording.

# Audience and viewing context

## Audience

Students or office workers feeling pressure to stay productive.

## Viewing context

A poster encountered briefly in a shared indoor space.

# Requirements and constraints

## Hard constraints

Avoid unsupported factual, health, or society-wide claims.

## Tone

Inviting rather than blaming or patronizing.

## Exact visible copy

Not specified.

## Additional requirements and context

### visual_style

International Typographic Style (Swiss Style). Use a clear modular grid, asymmetric but balanced composition, sans-serif typography, and a strong typographic hierarchy. Use generous negative space, a predominantly black-and-white palette, and at most one restra

## 4. Designer: propose, clarify, or review

Enable one paid designer call after reading the preview. No image call follows automatically.
If status is `needs_clarification` or `needs_sources`, supply the missing information
and return to the designer preview. `image_spec` remains null until the proposal is ready.
After self-review, inspect `revision_summary` and stop if no useful change is justified.
Each call is saved as a separate attempt, including failures and repeated proposals.


In [7]:
RUN_DESIGNER = True  # Set to False to skip the designer call and only preview the request.
if RUN_DESIGNER:
    if request_token(current_designer_request()) != designer_preview_token:
        raise ValueError("Inputs changed. Preview the designer request again.")
    if run_dir is None:
        run_dir = start_run(brief, original_user_request=original_user_request, output_dir=OUTPUT_DIR)
    proposal_record = design_round(
        designer_request, approved_token=designer_preview_token,
        run_dir=run_dir, history=rounds,
    )
    proposal_records.append(deepcopy(proposal_record))
    display(Markdown(render_design_rationale(proposal_record["proposal"])))
    print(f"Saved designer attempt: {proposal_record['folder']}")
else:
    print("Designer call is off. Preview the request, then enable when ready.")


# Design rationale

## Proposal status

ready

## 1. Communication purpose and context

<!-- Explain the purpose, audience, and viewing conditions relevant to this design. -->
The poster should offer students and office workers a brief, non-judgmental interruption to the visual and mental rhythm of continuous productivity. Because it will be encountered quickly indoors, the message must be legible at a glance and communicate through a simple, memorable typographic structure.

## 2. Core design concept

<!-- Introduce the overall idea and its principal elements before analyzing details. -->
Represent a pause as one deliberate open space inside an otherwise orderly sequence of typographic or modular units. The poster says “MAKE ROOM FOR A MOMENT.” A secondary line, “PAUSE, THEN CONTINUE.”, frames slowing down as temporary and compatible with returning to activity rather than as a rejection of productivity.

## 3. Signs and intended readings

<!-- Use a subsection for each important element. Identify its visible form and
referent, then explain each relevant symbolic, iconic, or indexical relationship
separately: grounds, intended reading, and limitations. Do not force all three
categories. Explain how the elements work together; do not claim audience effects
as observed facts before they have been tested. -->
### Open cell in a modular sequence
- **Referent:** A brief pause within a day of tasks, messages, meetings, or deadlines.
- **Relationship:** Primarily symbolic, with an iconic schematic quality. A sequence of repeated modules resembles a simplified plan, checklist, or progress system; the missing module conventionally suggests interruption or a deliberate gap.
- **Grounds:** Repetition establishes an expected rhythm, while one clearly bounded empty cell makes the break perceptible without depicting a literal person or workplace.
- **Intended reading:** “There can be a deliberate gap in the sequence.” The exact meaning of the gap remains somewhat open, but the nearby wording anchors it as a pause rather than an error.

### Main wording: “MAKE ROOM FOR A MOMENT.”
- **Referent:** Permission to create a short interval without demanding a large lifestyle change.
- **Relationship:** Symbolic, through ordinary language and imperative phrasing. “Room” also has a mild spatial echo, linking the wording to the visible negative space.
- **Grounds:** The wording is concise, invitational, and avoids claiming that constant productivity is universally harmful or that the viewer is doing something wrong.
- **Intended reading:** “Allow a small pause now.”

### Secondary wording: “PAUSE, THEN CONTINUE.”
- **Referent:** A pause as a temporary, self-directed interruption rather than an abandonment of responsibilities.
- **Relationship:** Symbolic and sequential. The two verbs establish a simple before-and-after order.
- **Grounds:** The comma and line break can make the pause visible in the reading rhythm; “continue” reduces the risk that the poster will feel unrealistic or patronizing to people who must return to work.
- **Intended reading:** “You may stop briefly and resume when you choose.” This remains an invitation, not a factual or health-related instruction.

### Restrained accent square
- **Referent:** The selected moment of interruption or attention.
- **Relationship:** Symbolic rather than inherently semantic; a small contrasting color block conventionally attracts attention and marks a point of emphasis.
- **Grounds:** Its meaning is established by its alignment with the open cell and wording, not by assuming that the color itself universally means rest.
- **Intended reading:** “Notice this gap.” The accent should remain subordinate to the typography.

## 4. Why this concept fits the task

<!-- Connect the concept to the communication purpose, audience, and constraints. -->
The open-cell concept makes the communication visible before the wording is fully read: an orderly sequence is present, but one space is intentionally protected. This suits a brief indoor viewing context because the structure can be understood at a distance and the message can be read in seconds. The language is direct but not blaming, and “then continue” acknowledges the audience’s practical need to return to study or work. Avoiding literal clocks, tired faces, phones, or office scenes keeps the poster focused and prevents the pause from being reduced to a specific personal or social diagnosis.

## 5. Visual style and art direction

<!-- Derive the proposed style from the preceding signs and intended readings.
Explain which relationships its visual qualities support and how. State the direction,
its defining visual qualities, and what may vary. Distinguish user-specified,
proposed for review, and explicitly confirmed directions, citing supplied evidence
for confirmation. A historical style label is optional; concrete qualities are not. -->
### Style or reference direction
The user has specified International Typographic Style (Swiss Style) as the visual direction. The proposal adopts its clear modular grid, asymmetric but balanced composition, sans-serif typography, strong hierarchy, generous negative space, predominantly black-and-white palette, and restrained use of one accent color.

### How it supports the signs
The modular grid makes the repeated sequence and its open cell precise and immediately legible. Asymmetric placement creates a controlled interruption without visual disorder. Generous negative space gives the pause a physical presence, supporting the wording’s spatial metaphor. Sans-serif typography and strong scale contrast ensure the invitation survives brief viewing in a shared indoor space.

### Defining visual qualities
- Portrait format with a visible underlying grid.
- Large black uppercase headline aligned to the grid.
- Small secondary line with clear separation and ample breathing room.
- A row or column of repeated black modules interrupted by one empty cell.
- One restrained muted red accent square, used only at the interruption point.
- Flat black, white, and accent color; no gradients, shadows, illustrations, or ornamental textures.
- Asymmetry balanced by consistent alignment and measured margins.

### Allowed variation and boundaries
The sequence may be horizontal or vertical, and the open cell may sit slightly off-center, provided it remains unmistakable and aligned with the typographic hierarchy. The exact scale of the modules may vary to suit the portrait format. The accent may be muted red or another single restrained accent color, but it must remain limited to the pause marker. No variation may turn the gap into clutter, a decorative pattern, or an apparent printing error.

### Confirmation status
The International Typographic Style requirements are explicitly supplied in the user brief and are therefore treated as design constraints. The open-cell motif, proposed wording, muted accent square, and exact composition are designer proposals for review; they have not been separately confirmed by the user. No approval is inferred from this ready status.

## 6. Visual decisions and their implementation

<!-- Explain composition, typography, color, imagery, and hierarchy where relevant.
Connect each concrete choice to its sign relationship, the supporting style quality,
and the image instruction that implements it. Treat supplied style requirements as
constraints from the outset; revisit signs and style together if they conflict. Keep the
explanation understandable without opening the image prompt or machine records. -->
### Composition
The open-cell sequence carries the pause concept, while the asymmetric grid placement prevents the poster from feeling like a generic productivity checklist. In the image specification, the sequence is placed in the upper-middle or middle field with generous surrounding white space; this preserves the relationship between regular rhythm and deliberate interruption.

### Typography
The headline must be the first readable element because the audience may pass by quickly. A large neutral grotesk sans-serif in uppercase supports the Swiss requirement and gives “ROOM” or “MOMENT” enough visual weight without decorative treatment. The secondary line is smaller but clearly legible and separated by space, preserving the pause in the reading rhythm.

### Visible copy
The exact proposed wording is limited to two short statements so the concept is not diluted: “MAKE ROOM FOR A MOMENT.” and “PAUSE, THEN CONTINUE.” The wording must remain unchanged during image preparation because its order establishes the intended temporary pause.

### Visual treatment
Flat black modules and typography establish the expected sequence; the white open cell makes the interruption visible; the single accent square directs attention without assigning a universal meaning to the color. These relationships must survive stylistic variation even if the sequence orientation or module scale changes.

### Allowed variation and exclusions
Variation is limited to grid orientation, module scale, and restrained accent choice. Decorative imagery, gradients, shadows, extra icons, and dense copy would weaken the open-space reading and are excluded.

## 7. Alternatives and tradeoffs

- A more minimal version using only “MAKE ROOM FOR A MOMENT.” would create a quieter poster, but it would give less reassurance that pausing is compatible with continuing work.
- A clock with missing segments could communicate interruption more literally, but it risks making the message about time management rather than permission to pause and would introduce a more illustrative element than the specified Swiss direction requires.

## 8. Assumptions and possible misreadings

### Assumptions behind the proposal

- The poster may use proposed wording because no exact visible copy was supplied.
- The poster is intended to be understood without a logo, campaign attribution, or additional explanatory body text.
- A muted red accent is acceptable as the one restrained accent color permitted by the style brief.

### Uncertainties and possible misreadings

- Some viewers may initially read the open module as a missing item or design error; the nearby wording and accent marker must make its intentionality clear.
- “Pause, then continue.” may be read as an instruction rather than an invitation, although its neutral wording and spacious treatment should soften that effect.
- The phrase “a moment” does not specify a duration, which preserves flexibility but leaves the exact pause open to interpretation.

## 9. How to review the result

<!-- Distinguish visible execution checks from questions requiring audience feedback. -->
- Can the headline be read immediately from a brief passing view?
- Is the open cell visibly intentional rather than mistaken for a missing or damaged element?
- Does the sequence communicate a temporary interruption without requiring a literal illustration?
- Does the poster feel inviting rather than accusatory, moralizing, or patronizing?
- Is the black-and-white Swiss structure dominant, with no more than one restrained accent color?
- Is there enough negative space for the pause to feel visually present?
- Are the two copy lines exactly reproduced and clearly hierarchized?

## 10. Review observations and changes

<!-- For an initial proposal, state that no generated result has been reviewed.
For a revision, connect observations to the intended design and explain changes. -->
No explanation supplied.

## 11. Information needed to proceed

### Required sources

None listed.

### Questions to resolve

None listed.


Saved designer attempt: D:\DH-Research\graphic-design-helper\experiments\pilot1\outputs\run_02\rounds\01\designer\attempt_01


## 5. Review two deliverables: design rationale and image prompt

Read the design rationale in order: purpose and concept, signs and intended readings,
why the concept fits, the style that supports those relationships, and concrete visual
implementation. Review uncertainties and distinguish visible checks from audience feedback.
User-specified styles constrain sign selection from the outset; signs and style may
be revised together. When changing either, update the explanation and image instructions
together so the intended relationships remain explicit.
It is saved as `design-rationale.md` alongside the designer's `proposal.json`.

The editable image specification below is the second deliverable. Its assembled
prompt is what the image model receives; the design explanation is not sent to it.
You may edit `image_spec` without changing the saved model proposal. If an edit changes
the concept or its sign relationships, revisit the design explanation before approval.
Rerun the separate preview cell after edits. Reinitializing the draft discards unsaved edits.


In [8]:
image_spec = deepcopy(proposal_record["proposal"]["image_spec"]) if "proposal_record" in globals() else None
image_settings = {
    "model": os.getenv("OPENAI_IMAGE_MODEL", "gpt-image-2"),
    "size": "1024x1536", "quality": "medium",
}
pending_image = rounds[-1] if rounds and rounds[-1].get("status") in {"requested", "failed", "interrupted"} else None
if pending_image is not None:
    image_spec = deepcopy(pending_image["image_spec"])
    image_settings = deepcopy(pending_image["settings"])
reviewed_token = None

def current_research():
    if "proposal_record" not in globals():
        raise ValueError("Request a design proposal first.")
    inputs = proposal_record["designer_input"]
    if brief != inputs["brief"] or original_user_request != inputs["original_user_request"]:
        raise ValueError("The user input changed. Request a new proposal.")
    if pending_image is not None:
        return deepcopy(pending_image["research"])
    return deepcopy({"brief": brief, "original_user_request": original_user_request,
                     "proposal": proposal_record["proposal"], "designer_record": proposal_record})

print(json.dumps(image_spec, ensure_ascii=False, indent=2))


{
  "communication_objective": "Invite students or office workers to allow a brief pause within a busy day, presenting slowing down as a temporary and self-directed interruption before continuing.",
  "audience_and_context": "Students or office workers feeling pressure to stay productive; a portrait poster encountered briefly in a shared indoor space. It must be legible at a glance and from several steps away.",
  "visible_copy": [
    "MAKE ROOM\nFOR A MOMENT.",
    "PAUSE, THEN CONTINUE."
  ],
  "composition": "Portrait poster with a clear modular grid and generous white margins. Place the large headline in an asymmetric but balanced block in the upper-left or left-center area. Place a horizontal sequence of evenly spaced black rectangular modules across the middle or lower-middle area, interrupted by one clearly bounded empty white cell. Align the empty cell with the typographic grid and place a small restrained accent square immediately beside or within the interruption point. Plac

In [9]:
# Rerun this cell after editing image_spec or the image prompt templates.
image_prompt_preview = compose_prompt(image_spec) if image_spec is not None else None
image_preview_context = request_token([image_spec, current_research(), image_settings]) if image_spec is not None else None
display(Markdown("# Image generation prompt\n\n" + image_prompt_preview)) if image_prompt_preview is not None else print("No ready image specification. Review the design rationale first.")
print(image_settings)


# Image generation prompt

# Rendering instructions

# Image rendering instructions

Render the poster described below. Only the strings in Exact visible text are intended
to appear in the artwork. Preserve their wording, punctuation, language, and intentional
line breaks. The JSON quotes and list syntax are delimiters, not artwork text.
An empty list means no visible text. Do not print instructions, section labels, or the
communication objective. Follow the composition, typography, visual treatment,
allowed variation, and exclusions. These instructions do not prescribe a default style.

Implement the direction described in Visual style and treatment together with the
composition and typography. Use the specified visual qualities to interpret any
style name; do not add stereotypical motifs, colors, or decoration merely because
a movement is named. Stay within Allowed variation and preserve the stated visual
character. Style labels and art-direction descriptions are instructions, not artwork text.

Preserve the specified elements and their spatial and hierarchical relationships
when applying the style. A style label does not authorize replacing those elements,
rearranging the message, or adding motifs. Use only the stated variation; execute
the supplied design rather than inventing a new concept to match a style.


# Communication objective

Invite students or office workers to allow a brief pause within a busy day, presenting slowing down as a temporary and self-directed interruption before continuing.

# Intended audience and viewing context

Students or office workers feeling pressure to stay productive; a portrait poster encountered briefly in a shared indoor space. It must be legible at a glance and from several steps away.

# Exact visible text

[
  "MAKE ROOM\nFOR A MOMENT.",
  "PAUSE, THEN CONTINUE."
]

# Visual composition

<!-- Specify the elements and relationships that the style must preserve. -->
Portrait poster with a clear modular grid and generous white margins. Place the large headline in an asymmetric but balanced block in the upper-left or left-center area. Place a horizontal sequence of evenly spaced black rectangular modules across the middle or lower-middle area, interrupted by one clearly bounded empty white cell. Align the empty cell with the typographic grid and place a small restrained accent square immediately beside or within the interruption point. Place the secondary line below the headline with ample vertical separation. Keep the composition quiet, spacious, and flat, with no crowded image area.

# Typography

Use a clean neutral grotesk sans-serif, uppercase, with strong typographic hierarchy. Set “MAKE ROOM” on the first line and “FOR A MOMENT.” on the second line, large and bold, with tight but readable line spacing. Set “PAUSE, THEN CONTINUE.” smaller in regular or medium weight below the headline. Use precise left alignment to the grid, generous tracking where appropriate, and high black-on-white contrast. No decorative lettering, outlined type, script, or warped typography.

# Visual style and treatment

International Typographic Style / Swiss-inspired execution: flat black typography and modules on a white ground, asymmetric grid alignment, measured margins, precise spacing, and strong negative space. Use one restrained muted red accent square only at the intentional gap in the module sequence. The repeated modules should resemble a simple visual rhythm or progress sequence; the single open cell must read as a deliberate pause. Use crisp edges, plain paper-like white, solid black, and no gradients, shadows, photographic imagery, icons, ornamental patterns, or unnecessary texture.

# Allowed variation

[
  "The repeated module sequence may run horizontally or vertically while remaining aligned to the grid.",
  "The open cell may be slightly off-center but must remain clearly visible and connected to the accent marker.",
  "The accent may be muted red or one similarly restrained single accent color.",
  "Module dimensions and exact headline scale may adjust to maintain legibility in the portrait format.",
  "Preserve the exact wording, the open-cell interruption, the strong hierarchy, and the generous negative space in every variation."
]

# Exclusions

[
  "No extra visible copy, logos, captions, statistics, or factual claims.",
  "No literal clocks, phones, office scenes, people, tired faces, coffee cups, plants, or landscape imagery.",
  "No gradients, drop shadows, bevels, glow effects, 3D rendering, decorative borders, or ornamental clutter.",
  "No multiple accent colors; keep the palette predominantly black and white.",
  "Do not make the open cell look accidental, damaged, noisy, or like a conventional loading spinner.",
  "Do not use cramped typography, centered symmetry, excessive visual density, or an aggressive warning tone."
]


{'model': 'gpt-image-2', 'size': '1024x1536', 'quality': 'medium'}


In [10]:
MARK_PROMPT_REVIEWED = True
if MARK_PROMPT_REVIEWED:
    if image_prompt_preview is None:
        raise ValueError("Resolve clarification or sources, then preview a ready specification.")
    if request_token([image_spec, current_research(), image_settings]) != image_preview_context:
        raise ValueError("Draft, rationale, or settings changed. Preview again.")
    reviewed_token = review_token(image_spec, current_research(), image_settings,
                                  image_prompt=image_prompt_preview)
    print("Complete image request and research marked reviewed.")


Complete image request and research marked reviewed.


## 6. Generate the poster

Enable one generation after inspecting the proposed prompt. This experiment retains the
limit of three image calls in total, including failed calls and explicit retries. Each request saves its proposal, prompt and image.
The next poster is generated from the updated text prompt; this notebook does not edit the
previous image's pixels. The previous image is visual input to the designer's self-review.

If a call is interrupted, its records say `interrupted` with an unknown remote outcome.
No retry occurs automatically. Restore the run in Section 2, then use Section 5 to
preview and review the saved request. Set `RETRY_IMAGE = True`, enter `RETRY_REASON`,
and enable `RUN_GENERATION` to send one new call in the same round. A retry preserves
the earlier attempt and uses a new `attempt_<number>` directory. The request must
remain unchanged, and successful rounds cannot be retried through this switch.
A record still marked `requested` after a kernel crash needs inspection before it
can be classified as interrupted; it is never retried automatically.


In [11]:
RUN_GENERATION = True
RETRY_IMAGE = False
RETRY_REASON = ""

if RUN_GENERATION:
    if reviewed_token is None:
        raise ValueError("Run the image prompt preview and MARK_PROMPT_REVIEWED cell first.")
    research = current_research()
    revision_decision = rounds[-1]["revision"] if RETRY_IMAGE and rounds else {
        "reason": research["proposal"]["revision_summary"] if rounds else "Initial proposal",
    }
    print("Waiting for the image response. Interrupting stops local waiting; the remote outcome may remain unknown.", flush=True)
    try:
        active_round = generate_round(
            image_spec, research, image_settings,
            approved_token=reviewed_token, history=rounds, revision=revision_decision,
            output_dir=OUTPUT_DIR, retry=RETRY_IMAGE, retry_reason=RETRY_REASON,
        )
        print(f"Saved round {active_round['round']} to {active_round['folder']}")
    finally:
        reviewed_token = None
        RUN_GENERATION = False
        RETRY_IMAGE = False
else:
    print("Image generation is off.")


Waiting for the image response. Interrupting stops local waiting; the remote outcome may remain unknown.
Saved round 1 to D:\DH-Research\graphic-design-helper\experiments\pilot1\outputs\run_02\rounds\01


In [ ]:
image_paths = [r["image_path"] for r in rounds if r.get("image_path")]
labels = [f"Round {r['round']}" for r in rounds if r.get("image_path")]
if image_paths:
    display(compare_images(image_paths, labels))


## 7. Look, reconsider, repeat

Return to Section 3 for self-review of the latest poster against the original brief.
Keep successful choices and stop when no useful change remains. The limit is three
image calls, including failed or uncertain calls and explicit retries; reaching it does not establish success.
A model's reading is a design judgment, not evidence of audience reception.

New records are grouped under `outputs/run_<number>/rounds/<number>/`. Each `designer/`
and `image/` directory contains separate `attempt_01`, `attempt_02`, ... folders with `request.json`,
`prompt.md`, and `response.json`. Designer attempts save `proposal.json`; image attempts
save the reviewed `image-spec.json` and `image.png`. The original proposal is preserved.

Use the experiment-selection cell in Section 2 after a restart. Historical outputs remain unchanged and
can be inspected directly; do not rerun them just to adopt the new record structure.


## 8. Optional audience readings

Before explaining the intention, ask: What did you notice first? What do you think this is saying?
Who seems to be speaking? Does it invite you to do anything? Record unexpected readings too.
These are exploratory observations, not evidence of long-term behavior change.

Store participants' words separately from your interpretation. Avoid identifying information.

For each element, compare the intended relationship with the reading people actually describe.
Ask about the imagery and arrangement in ordinary language; viewers need not know Peircean terms.
If something looks like evidence, ask what they think it documents. Do not supply that answer first.


In [ ]:
observations = []  # For each response: variant, anonymous label, verbatim wording, researcher interpretation.
reflection = {
    "what_worked": "",
    "unexpected_readings": "",
    "design_changes_to_try": "",
    "semiotic_assumptions_to_revisit": "",
    "relations_supported_or_challenged_by_responses": "",
    "source_or_evidence_confusions": "",
    "limitations": "No audience responses recorded in this reflection.",
}


## 9. Save a snapshot

Set `SAVE_SNAPSHOT` to `True` when you want to save the current state. Each save creates a new `outputs/snapshot_<number>/` folder
with notes, image copies, and hashes. Include only non-secret settings; never pass environment
variables, API keys, or client objects. The saved status distinguishes planning from collected evidence.


In [ ]:
SAVE_SNAPSHOT = False
if SAVE_SNAPSHOT:
    run = save_experiment({
        "status": "exploration" if rounds else "planning",
        "brief": brief, "designer_request": designer_request,
        "proposal_records": proposal_records,
        "image_spec_draft": image_spec, "rounds": rounds,
        "audience_observations": observations, "reflection": reflection,
    }, [r["image_path"] for r in rounds if r.get("image_path")], OUTPUT_DIR)
    print(f"Saved: {run}")
